In [1]:
# Installa le librerie necessarie
!pip install -q gradio transformers torch pillow


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os
import joblib
import torch
import pandas as pd
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# --- 1. CONFIGURAZIONE DISPOSITIVO ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device selezionato: {device}")

# --- 2. PERCORSI FILE ---
BASE_PATH = "/content/drive/MyDrive/Tesi"
model_path = os.path.join(BASE_PATH, "modello_voting_ensemble.pkl")
scaler_path = os.path.join(BASE_PATH, "scaler.pkl")

# --- 3. CARICAMENTO CLASSIFICATORE SKLEARN ---
print(f"📦 Caricamento classificatore da {model_path}...")
clf_model = joblib.load(model_path)
print("✅ Classificatore caricato!")

# --- 4. CARICAMENTO SCALER ---
print(f"📦 Caricamento scaler da {scaler_path}...")
scaler = joblib.load(scaler_path)
print("✅ Scaler caricato!")

# --- 5. CARICAMENTO CLIP ---
print("📦 Caricamento CLIP (openai/clip-vit-large-patch14)...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model.eval()
print("✅ CLIP caricato!")

print("\n🎉 Tutti i modelli sono pronti!\n")


✅ Device selezionato: cpu
📦 Caricamento classificatore da /content/drive/MyDrive/Tesi/modello_voting_ensemble.pkl...
✅ Classificatore caricato!
📦 Caricamento scaler da /content/drive/MyDrive/Tesi/scaler.pkl...
✅ Scaler caricato!
📦 Caricamento CLIP (openai/clip-vit-large-patch14)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

✅ CLIP caricato!

🎉 Tutti i modelli sono pronti!



In [4]:
def extract_features(image):
    try:
        image = image.convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        with torch.no_grad():
            # Usa vision_model + proiezione finale
            vision_outputs = clip_model.vision_model(pixel_values=pixel_values)
            image_embeds = vision_outputs[1]  # pooled_output
            # Applica la proiezione visuale per ottenere 768 dim
            image_embeds = clip_model.visual_projection(image_embeds)

        features = image_embeds.cpu().numpy().flatten()

        if len(features) != 768:
            print(f"❌ Dimensione errata: {len(features)}")
            return None

        cols = [f"feature_{i}" for i in range(768)]
        return pd.DataFrame([features], columns=cols)

    except Exception as e:
        print(f"❌ Errore: {e}")
        return None


In [5]:
def predict_image(image):
    """
    Funzione principale chiamata da Gradio.
    Input: immagine PIL
    Output: (testo risultato, dizionario probabilità, HTML dettagli)
    """
    if image is None:
        return "❌ Nessuna immagine caricata", {}, ""

    print("🔄 Analisi in corso...")

    # --- 1. ESTRAI FEATURES ---
    df_features = extract_features(image)
    if df_features is None:
        return "❌ Errore nell'estrazione delle features", {}, ""

    # Verifica dimensioni
    if df_features.shape[1] != 768:
        return f"❌ Errore: estratte {df_features.shape[1]} features invece di 768", {}, ""

    # --- 2. APPLICA SCALER ---
    try:
        # Se lo scaler ha feature_names_in_, usa DataFrame per evitare warning
        if hasattr(scaler, "feature_names_in_"):
            X_scaled = scaler.transform(df_features)
        else:
            X_scaled = scaler.transform(df_features.to_numpy())
    except Exception as e:
        print(f"⚠️ Warning scaler: {e}")
        X_scaled = scaler.transform(df_features.to_numpy())

    # --- 3. PREDIZIONE ---
    try:
        # Probabilità classe 1 (AI)
        if hasattr(clf_model, "predict_proba"):
            probs = clf_model.predict_proba(X_scaled)[0]
            prob_real = float(probs[0])
            prob_ai = float(probs[1])
        else:
            # Fallback se il modello non supporta probabilità
            pred = clf_model.predict(X_scaled)[0]
            prob_ai = 1.0 if pred == 1 else 0.0
            prob_real = 1.0 - prob_ai

        # Determina classe vincente
        is_ai = prob_ai >= 0.5

        # --- OUTPUT 1: Testo semplice ---
        if is_ai:
            label_text = "🤖 IMMAGINE GENERATA DA AI"
            color = "#ffdddd"  # Rosso chiaro
            border_color = "#ff0000"
        else:
            label_text = "📷 IMMAGINE REALE"
            color = "#ddffdd"  # Verde chiaro
            border_color = "#008000"

        # --- OUTPUT 2: Dizionario probabilità per grafico ---
        prob_dict = {
            "Reale": prob_real,
            "AI": prob_ai
        }

        # --- OUTPUT 3: HTML con dettagli ---
        html_output = f"""
        <div style="background-color: {color}; padding: 20px; border-radius: 10px;
                    border: 3px solid {border_color}; text-align: center; margin-top: 20px;">
            <h2 style="margin: 0; color: #333;">
                {'🤖 IMMAGINE AI' if is_ai else '📷 IMMAGINE REALE'}
            </h2>
            <hr style="border-top: 1px solid {border_color}; opacity: 0.5;">
            <p style="font-size: 18px; margin: 10px 0;">
                <b>Confidenza AI:</b> {prob_ai*100:.2f}%
            </p>
            <p style="font-size: 18px; margin: 10px 0;">
                <b>Confidenza Reale:</b> {prob_real*100:.2f}%
            </p>
        </div>
        """

        print(f"✅ Analisi completata: {'AI' if is_ai else 'Reale'}")
        return label_text, prob_dict, html_output

    except Exception as e:
        print(f"❌ Errore durante predizione: {e}")
        return f"❌ Errore: {str(e)}", {}, ""


In [ ]:
import gradio as gr

# Chiudi eventuali istanze precedenti di Gradio
gr.close_all()

# --- TESTI INTERFACCIA ---
title = "🔍 AI Image Detector - Rilevatore Immagini AI"

description = """
### 🎓 Progetto di Tesi Universitario

Questo strumento utilizza:
- **CLIP** (OpenAI) per l'estrazione delle features
- **Voting Classifier** (SVM + Random Forest + MLP) per la classificazione

Carica un'immagine per scoprire se è **reale 📷** o **generata da AI 🤖**
"""

article = """
<p style="text-align: center; color: #666; margin-top: 20px;">
⚠️ <b>Disclaimer:</b> Nessun algoritmo è infallibile.
Utilizza i risultati come supporto, non come verità assoluta.
</p>
"""

# --- CREAZIONE INTERFACCIA ---
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="📤 Carica un'immagine"),
    outputs=[
        gr.Textbox(label="🎯 Risultato", show_label=True),
        gr.Label(num_top_classes=2, label="📊 Probabilità"),
        gr.HTML(label="📋 Dettagli Analisi")
    ],
    title=title,
    description=description,
    article=article,
    theme=gr.themes.Soft(),
    allow_flagging="never"
)

# --- LANCIO ---
print("\n🚀 Avvio interfaccia Gradio...\n")
interface.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(



🚀 Avvio interfaccia Gradio...

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3f691e345497ab879d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🔄 Analisi in corso...
✅ Analisi completata: Reale
🔄 Analisi in corso...
✅ Analisi completata: AI
🔄 Analisi in corso...
✅ Analisi completata: Reale
🔄 Analisi in corso...
✅ Analisi completata: Reale
🔄 Analisi in corso...
✅ Analisi completata: Reale
🔄 Analisi in corso...
✅ Analisi completata: Reale
🔄 Analisi in corso...
✅ Analisi completata: AI
🔄 Analisi in corso...
✅ Analisi completata: AI
🔄 Analisi in corso...
✅ Analisi completata: AI
🔄 Analisi in corso...
✅ Analisi completata: Reale
🔄 Analisi in corso...
✅ Analisi completata: AI
🔄 Analisi in corso...
✅ Analisi completata: AI
